# Step 1 - Building & Exporting the Model

## Insurance Premium Category Predictor

We build a **Random Forest Classifier** to predict whether a customer falls into
`Low`, `Medium`, or `High` insurance premium category based on demographic and
lifestyle features.

**Target variable:** `insurance_premium_category`
**Algorithm:** Random Forest (ensemble of decision trees)
**Output:** Trained pipeline exported as `model.pkl`

In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

## Load Data

Read the CSV file containing raw insurance customer data.
Each row is one customer. The last column is our target label.

In [ ]:
df = pd.read_csv('../data/insurance_data.csv')

## Preview Data

`df.sample(5)` shows 5 random rows — better than `.head()` for spotting
distribution across categories.

In [3]:
df.sample(5)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
293,24,92.3,1.68,11.82714,False,Bangalore,freelancer,Medium
1,69,102.4,1.57,12.81611,True,Pune,salaried_private,High
114,41,70.8,1.78,10.55416,True,Mumbai,freelancer,High
292,49,128.1,1.65,5.63341,True,Kolkata,freelancer,High
86,53,72.4,1.74,28.08320,True,Bangalore,salaried_private,High


## Feature Engineering

Raw columns alone are not enough. We derive 4 new features that carry
stronger predictive signal for insurance risk:

| Feature | Logic |
|---|---|
| `bmi` | weight / height² |
| `age_group` | binned age into young / adult / middle_aged / senior |
| `lifestyle_risk` | combined smoker + BMI rule |
| `city_tier` | tier-1 / tier-2 / other city classification |

In [4]:
df_feat = df.copy()

### Feature 1: BMI

Body Mass Index = weight (kg) / height (m)²

Standard WHO measure. High BMI correlates with higher health risk,
which insurers price into premiums.

In [5]:
# Feature 1: BMI
df_feat["bmi"] = df_feat["weight"] / (df_feat["height"] ** 2)

### Feature 2: Age Group

Age as a raw number is continuous. Binning it into groups helps the model
learn non-linear age effects without needing more trees.

- `young` → age < 25
- `adult` → 25 ≤ age < 45
- `middle_aged` → 45 ≤ age < 60
- `senior` → age ≥ 60

In [6]:
# Feature 2: Age Group
def age_group(age):
    if age < 25:
        return "young"
    elif age < 45:
        return "adult"
    elif age < 60:
        return "middle_aged"
    return "senior"

df_feat["age_group"] = df_feat["age"].apply(age_group)

### Feature 3: Lifestyle Risk

Combines two risk signals — smoking and BMI — into one categorical feature.

| Condition | Risk Level |
|---|---|
| Smoker AND BMI > 30 | high |
| Smoker OR BMI > 27 | medium |
| Otherwise | low |

This captures interaction effects that a single feature would miss.

In [7]:
# Feature 3: Lifestyle Risk
def lifestyle_risk(row):
    if row["smoker"] and row["bmi"] > 30:
        return "high"
    elif row["smoker"] or row["bmi"] > 27:
        return "medium"
    else:
        return "low"

df_feat["lifestyle_risk"] = df_feat.apply(lifestyle_risk, axis=1)

### Feature 4: City Tier

Indian cities are tiered by cost of living and healthcare access.
Insurers adjust premiums based on city tier.

- **Tier 1** → Mumbai, Delhi, Bangalore, Hyderabad, Chennai, Kolkata (return 1)
- **Tier 2** → Pune, Ahmedabad, Jaipur, Lucknow (return 2)
- **Others** → return 3

In [8]:
# Feature 4: City Tier
tier_1_cities = ["Mumbai", "Delhi", "Bangalore", "Hyderabad", "Chennai", "Kolkata"]
tier_2_cities = ["Pune", "Ahmedabad", "Jaipur", "Lucknow"]

def city_tier(city):
    if city in tier_1_cities:
        return 1
    elif city in tier_2_cities:
        return 2
    else:
        return 3

df_feat["city_tier"] = df_feat["city"].apply(city_tier)

## Drop Raw Columns, Keep Engineered Features

We drop the original columns (`age`, `weight`, `height`, `smoker`, `city`)
since they have been transformed into better features.
Final feature set used for modelling:

`income_lpa`, `occupation`, `bmi`, `age_group`, `lifestyle_risk`, `city_tier`

In [9]:
df_feat.drop(columns=["age", "weight", "height", "smoker", "city"])[
    ["income_lpa", "occupation", "bmi", "age_group", "lifestyle_risk", "city_tier", "insurance_premium_category"]
].sample(3)

,income_lpa,occupation,bmi,age_group,lifestyle_risk,city_tier,insurance_premium_category
267,31.60833,salaried_private,44.346059,senior,medium,1,High
188,16.33257,self_employed,34.740484,middle_aged,high,1,High
235,14.48006,self_employed,26.234754,senior,medium,2,High


## Select Features and Target

- **X** → input features the model learns from
- **y** → target label the model predicts

In [10]:
# Select features and target
X = df_feat[["bmi", "age_group", "lifestyle_risk", "city_tier", "income_lpa", "occupation"]]
y = df_feat["insurance_premium_category"]

## Preprocessing: OneHotEncoder + Passthrough

Categorical features (`age_group`, `lifestyle_risk`, `occupation`, `city_tier`)
cannot be fed directly to a numeric model — they need to be encoded.

`ColumnTransformer` applies:
- `OneHotEncoder` → to categorical columns (converts to 0/1 dummy columns)
- `passthrough` → to numeric columns (`bmi`, `income_lpa`), keeps them as-is

In [11]:
# Define categorical and numeric features
categorical_features = ["age_group", "lifestyle_risk", "occupation", "city_tier"]
numeric_features = ["bmi", "income_lpa"]

# Create column transformer for OHE
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

## Build Pipeline

`Pipeline` chains preprocessing and model into one object.

Benefits:
- No data leakage — preprocessing fits only on train set
- Single object to save and deploy
- `pipeline.predict()` handles encoding automatically at inference time

In [12]:
# Create pipeline with preprocessing and random forest classifier
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

## Train-Test Split and Model Fit

- 80% data for training, 20% held out for evaluation
- `random_state=1` ensures reproducibility
- `pipeline.fit()` runs OHE on X_train, then trains the Random Forest

In [13]:
# Split data and train model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat', OneHotEncoder(),
                                                  ['age_group',
                                                   'lifestyle_risk',
                                                   'occupation', 'city_tier']),
                                                 ('num', 'passthrough',
                                                  ['bmi', 'income_lpa'])])),
                ('classifier', RandomForestClassifier(random_state=42))])

## Evaluate Model

`accuracy_score` gives the percentage of correct predictions on unseen test data.

A score of ~0.9 means the model correctly classifies 9 out of 10 customers.

In [14]:
# Predict and evaluate
y_pred = pipeline.predict(X_test)
accuracy_score(y_test, y_pred)

0.85

## Export Model as Pickle File

`pickle` serializes the entire pipeline object to disk as `model.pkl`.

This file contains:
- The fitted OneHotEncoder (knows all categories from training)
- The trained Random Forest (all 100 decision trees)

In Step 2, a Flask/Streamlit app will load this file and call
`pipeline.predict()` on new user inputs without retraining.

In [ ]:
import pickle

# Save the trained pipeline using pickle
pickle_model_path = "../model/model.pkl"
with open(pickle_model_path, "wb") as f:
    pickle.dump(pipeline, f)